# Семинар 8 - потоки

На прошлом семинаре мы говорили про процессы, сегодня будем говорить про потоки. 
Можно сказать, что поток - это единица исполнения в Linux. Они отчасти похожи на процессы, но есть следующие нюансы:
* Потоки существуют в рамках процесса
* Из-за предыдущего пункта, у поток общее адресное пространство и отдельные стеки. В отличие от процессов, у которых отдельные адресные пространства
* Из-за предыдущего пункта и других особенностей, переключение исполнения между потоками происходит намного эффективнее, чем между процессами
* Из-за предыдущего пункта и других особенностей, в случае, когда вам надо "распараллелить" программу, следует использовать потоки
* В ядре процессы и потоки представлены одной и той же структурой `task_struct`

Для создания нового потока используется функция `pthread_create`:

```c
pthread_create(3)                                   Library Functions Manual                                  pthread_create(3)

NAME
       pthread_create - create a new thread

LIBRARY
       POSIX threads library (libpthread, -lpthread)

SYNOPSIS
       #include <pthread.h>

       int pthread_create(pthread_t *restrict thread,
                          const pthread_attr_t *restrict attr,
                          void *(*start_routine)(void *),
                          void *restrict arg);

DESCRIPTION
       The  pthread_create()  function starts a new thread in the calling process.  The new thread starts execution by invoking
       start_routine(); arg is passed as the sole argument of start_routine().
```

Эта функция принимает указатель на функцию `start_routine` - это и есть та функция, которая будет исполняться в новом потоке. Ей будет передан аргумент `arg`. Также можно передать набор аттрибутов нового потока:

```c
Thread attributes:
                   Detach state        = PTHREAD_CREATE_JOINABLE
                   Scope               = PTHREAD_SCOPE_SYSTEM
                   Inherit scheduler   = PTHREAD_INHERIT_SCHED
                   Scheduling policy   = SCHED_OTHER
                   Scheduling priority = 0
                   Guard size          = 4096 bytes
                   Stack address       = 0x40196000
                   Stack size          = 0x201000 bytes
```

После того, как поток окажется больше не нужен, нужно вызвать функцию `pthread_join`:

```c
pthread_join(3)                                     Library Functions Manual                                    pthread_join(3)

NAME
       pthread_join - join with a terminated thread

LIBRARY
       POSIX threads library (libpthread, -lpthread)

SYNOPSIS
       #include <pthread.h>

       int pthread_join(pthread_t thread, void **retval);

DESCRIPTION
       The  pthread_join()  function  waits for the thread specified by thread to terminate.  If that thread has already termi‐
       nated, then pthread_join() returns immediately.  The thread specified by thread must be joinable.
```

При это функция заблокирует исполнения вызываемого потока до тех пор, пока завершаемый поток не завершит исполнение.

### Посмотрим на простейшем примере:

In [1]:
!cat snippets/create-join/create-join.c 
!gcc snippets/create-join/create-join.c -o snippets/create-join/create-join.out

#include <pthread.h>
#include <stdio.h>

void* thread_func(void* arg) {
    printf("  Thread func started\n");
    printf("  Thread func finished\n");
    return NULL;
}

int main() {
    printf("Main func started\n");
    pthread_t thread;
    printf("Thread creating\n");
    pthread_create(&thread, NULL, thread_func, 0); // В какой-то момент будет создан поток и в нем вызвана функция
    // Начиная отсюда неизвестно в каком порядке выполняются инструкции основного и дочернего потока
    pthread_join(thread, NULL); // -- аналог waitpid. Второй аргумент -- указатель в который запишется возвращаемое значение
    printf("Thread joined\n");
    printf("Main func finished\n");
    return 0;
}

In [2]:
!./snippets/create-join/create-join.out 

Main func started
Thread creating
  Thread func started
  Thread func finished
Thread joined
Main func finished


### Запустим теперь несколько потоков друг за другом:

In [3]:
!cat snippets/multiple-create/multiple-create.c 
!gcc snippets/multiple-create/multiple-create.c -o snippets/multiple-create/multiple-create.out

#include <pthread.h>
#include <stdio.h>

static int N = 10;

void* thread_func(void* arg) {
    int id = *(int*)arg;
    printf("  Thread %d func started\n", id);
    printf("  Thread %d func finished\n", id);
    return NULL;
}

int main() {
    printf("Main func started\n");
    pthread_t thread[N];
    int       counter[N];
    for (int i = 0; i < N; ++i) {
        counter[i] = i;
        pthread_create(&thread[i], NULL, thread_func, &counter[i]);
    }
    for (int i = 0; i < N; ++i) {
        pthread_join(thread[i], NULL);
    }
    printf("Main func finished\n");
    return 0;
}

In [9]:
!./snippets/multiple-create/multiple-create.out 

Main func started
  Thread 0 func started
  Thread 0 func finished
  Thread 4 func started
  Thread 6 func started
  Thread 6 func finished
  Thread 2 func started
  Thread 2 func finished
  Thread 5 func started
  Thread 5 func finished
  Thread 3 func started
  Thread 3 func finished
  Thread 8 func started
  Thread 8 func finished
  Thread 9 func started
  Thread 9 func finished
  Thread 1 func started
  Thread 1 func finished
  Thread 4 func finished
  Thread 7 func started
  Thread 7 func finished
Main func finished


Видно, что потоки создаются и исполняются, не следуя какому-либо порядку. 

Это ставит следующую проблему: если несколько потоков работают с общим ресурсом (например, с общей областью памяти), у нас нет гарантий про то, как друг относительно друг друга расположены моменты времени, когда потоки работают с общим ресурсом. Например, если 1 поток присваивает значение в переменную, а другой из нее читает, то читатель может увидеть как значение до изменения, так и после. Более того, подобная ситуация вызывает undefined behavior, значит, читатель может прочитать вообще любое значение. 

На эту проблему можно смотреть с 2 точек зрения:
1. Можно попытаться объяснить такое поведение особенностью устройства процессора и памяти. Например, сказать, что в общем случае чтение и запись в переменную - это 3 операции (`ldr`, `mov`, `str`), и 2 потока могут их исполнить в произвольном порядке.
2. Можно опираться на аксиоматический подход, то есть на набор правил, который отвечает на вопрос, как связаны между собой исполнения разных потоков. 

Поскольку почти все современные языки программирования опираются на 2 подход, именно его стоит использовать при работе с ними. Первый же может дать первичную интуицию и мотивацию, но использовать его при написании Си кода, например, неправильно.

Итого, говоря более формально: *если несколько потоков работают над общим ресурсом, и хотя бы один из потоков изменяет этот ресурс, то все эти обращения к общему ресурсу нуждаются в синхронизации. Ситуация, при которой такие обращения не синхронизированы, называется data race и является UB.*

Синхронизация (упорядочивание) - это создание ограничений на исполнение мнопоточной программы, с целью усиления гарантий на возможные результаты этого исполнения.

In [33]:
!cat snippets/no-sync/no-sync.c 
!gcc snippets/no-sync/no-sync.c -o snippets/no-sync/no-sync.out

#include <stdio.h>
#include <pthread.h>
#include <stdint.h>
#include <inttypes.h>
#include <stdlib.h>

typedef struct {
    pthread_t thread;
    int* counter;
} thread_data;

static void* thread_func(void* arg) {
    thread_data* data_ptr = (thread_data*)arg;
    for (int i = 0; i < 30000; i++) {
        ++(*(data_ptr->counter));
    }
    return NULL;
}

int main() {
    const int THREADS_COUNT = 10;
    thread_data threads[THREADS_COUNT];
    int shared_counter = 0;
    for (int i = 0; i < THREADS_COUNT; i++) {
        threads[i].counter = &shared_counter;
        pthread_create(&threads[i].thread, NULL, thread_func,  (void*)&threads[i]);
    }
    for (int i = 0; i < THREADS_COUNT; i++) {
        pthread_join(threads[i].thread, NULL);
    }
    printf("%d\n", shared_counter);
    return 0;
}


In [29]:
!./snippets/no-sync/no-sync.out 

157883


Один из способов смотреть на исполнение многопоточных программ таков: можно представить, что истинно параллельного исполнения не существует, а вместо этого в каждый момент времени исполняется лишь 1 поток. При этом в любой момент времени, *когда это возможно*, он может быть снят с исполнения в пользу другого потока.

Такая модель называется sequential consistency model, и ее нам будет достаточно для работы на семинаре и выполнения домашних заданий. Вообще бывают и другие модели исполнения, для которых такое представление некорректно.

<img src="media/seq_const.png" alt="sequential consistency example" width="600" style="background-color:white;"/>

### Рассмотрим еще один пример многопоточной программы, в которой есть data race:

In [10]:
!cat snippets/faulty/faulty.c 
!gcc snippets/faulty/faulty.c -o snippets/faulty/faulty.out

#include <stdio.h>
#include <stdint.h>
#include <stddef.h>
#include <stdlib.h>
#include <pthread.h>

int64_t x;
int64_t y;
int64_t r1;
int64_t r2;

void* write_x_read_y(void* arg) {
  (void)arg;

  x = 1;
  r1 = y;

  return NULL;
}

void* write_y_read_x(void* arg) {
  (void)arg;

  y = 1;
  r2 = x;

  return NULL;
}

int main() {
  for (size_t i = 0;; ++i) {
    x = 0;
    y = 0;

    r1 = 0;
    r2 = 0;

    pthread_t t1;
    pthread_create(&t1, /*attr=*/NULL, &write_x_read_y, /*arg=*/NULL);
    pthread_t t2;
    pthread_create(&t2, /*attr=*/NULL, &write_y_read_x, /*arg=*/NULL);

    pthread_join(t1, /*thread_return=*/NULL);
    pthread_join(t2, /*thread_return=*/NULL);

    if (r1 == 0 && r2 == 0) {
      printf("Iteration #%lu: CPU is broken\n", i);
      abort();
    }
  }
}


In [35]:
!./snippets/faulty/faulty.out

Iteration #180: CPU is broken


В какой-то момент мы получили исполнение, которое невозможно в sequential consistency model:

<img src="media/faulty.png" alt="data race failing sequential consistency" width="600" style="background-color:white;"/>

В рассмотренной программе есть data race, а значит, она содержит undefined behavior. Поэтому мы получили результат, не соответствующий ожиданиям. 

Чтобы это исправить, используем простейший инструмент синхронизации из Си: `atomic`. Если мы используем `atomic` тип, то операции с ним будут атомарными, то есть наблюдатель всегда будет наблюдать объект или до применения операции, или после, и никогда в промежуточных состояниях. Говоря в парадигме seq const model, исполнение потока не сможет быть прервано в момент совершения атомарной операции.

В архитектуре x86 атомарными могут быть типы, у которых `sizeof` не больше 16 байт. Более того, правильно говорить не об атомарности типа, а об атомарности действия над памятью. Для упрощения повествования я позволю себе в дальнейшем использовать понятия атомарного типа.

In [14]:
!cat snippets/faulty/atomic.c 
!gcc snippets/faulty/atomic.c -o snippets/faulty/atomic.out

#include <stdio.h>
#include <stdint.h>
#include <stddef.h>
#include <stdlib.h>
#include <pthread.h>

_Atomic int64_t x;
_Atomic int64_t y;
_Atomic int64_t r1;
_Atomic int64_t r2;

void* write_x_read_y(void* arg) {
  (void)arg;

  x = 1;
  r1 = y;

  return NULL;
}

void* write_y_read_x(void* arg) {
  (void)arg;

  y = 1;
  r2 = x;

  return NULL;
}

int main() {
  for (size_t i = 0;; ++i) {
    x = 0;
    y = 0;

    r1 = 0;
    r2 = 0;

    pthread_t t1;
    pthread_create(&t1, /*attr=*/NULL, &write_x_read_y, /*arg=*/NULL);
    pthread_t t2;
    pthread_create(&t2, /*attr=*/NULL, &write_y_read_x, /*arg=*/NULL);

    pthread_join(t1, /*thread_return=*/NULL);
    pthread_join(t2, /*thread_return=*/NULL);

    if (r1 == 0 && r2 == 0) {
      printf("Iteration #%lu: CPU is broken\n", i);
      abort();
    }
  }
}


In [18]:
!./snippets/faulty/atomic.out

OSError: [Errno 5] Input/output error

Теперь, когда в этом примере используются атомарные типы, программа снова стала исполняться согласно seq const model. У атомиков есть целый набор атомарных операций: `load, store, fetch_[add/sub], exchange, CAS`.

Пример использования `fetch_add`:

In [34]:
!cat snippets/no-sync/no-sync-fix.c 
!gcc snippets/no-sync/no-sync-fix.c -o snippets/no-sync/no-sync-fix.out

#include <stdio.h>
#include <pthread.h>
#include <stdint.h>
#include <inttypes.h>
#include <stdlib.h>

typedef struct {
    pthread_t thread;
    _Atomic int* counter;
} thread_data;

static void* thread_func(void* arg) {
    thread_data* data_ptr = (thread_data*)arg;
    for (int i = 0; i < 30000; i++) {
        ++(*(data_ptr->counter));
    }
    return NULL;
}

int main() {
    const int THREADS_COUNT = 10;
    thread_data threads[THREADS_COUNT];
    int shared_counter = 0;
    for (int i = 0; i < THREADS_COUNT; i++) {
        threads[i].counter = &shared_counter;
        pthread_create(&threads[i].thread, NULL, thread_func,  (void*)&threads[i]);
    }
    for (int i = 0; i < THREADS_COUNT; i++) {
        pthread_join(threads[i].thread, NULL);
    }
    printf("%d\n", shared_counter);
    return 0;
}
snippets/no-sync/no-sync-fix.c:25:28: warning: incompatible pointer types assigning to '_Atomic(int) *' from 'int *' [-Wincompatible-pointer-types]
   25 |         threads[i].counter = 

In [32]:
!./snippets/no-sync/no-sync-fix.out

300000


[Deadlock Empire](https://deadlockempire.github.io/) - игра, чтобы попрактиковаться в seq cost модели.

### Есть и более высокоуровневые и удобные примитивы синхронизации
Пожалуй, самый распространенный из них - `mutex`.

Критическая секция - часть кода многопоточной программы, которую в каждый момент времени должен исполнять максимум 1 поток.

`mutex` (mutual exclusion) - инструмент многопоточной синхронизации, позвояющий добиться того, что часть кода в каждый момент времени будет исполняться максимум 1 потоком. У мьютекса есть 2 основных метода: `lock (aquire)` и `unlock (release)`.

Их семантика такова: `lock` блокирует исполнение текущего потока, если мьютекс уже взят другим потоком, до момента, когда текущий поток захватит его. `unlock` освобождает мьютекст, если текущий поток им до этого владел. 

Посмотрим использование мьютекса на примере:

In [21]:
!cat snippets/faulty/mutex.c 
!gcc snippets/faulty/mutex.c -o snippets/faulty/mutex.out

#include <stdio.h>
#include <stdint.h>
#include <stddef.h>
#include <stdlib.h>
#include <pthread.h>

int64_t x;
int64_t y;
int64_t r1;
int64_t r2;
pthread_mutex_t mutex;

void* write_x_read_y(void* arg) {
  (void)arg;

  pthread_mutex_lock(&mutex);
  x = 1;
  r1 = y;
  pthread_mutex_unlock(&mutex);

  return NULL;
}

void* write_y_read_x(void* arg) {
  (void)arg;

  pthread_mutex_lock(&mutex);
  y = 1;
  r2 = x;
  pthread_mutex_unlock(&mutex);

  return NULL;
}

int main() {
  pthread_mutex_init(&mutex, NULL);
  
  for (size_t i = 0;; ++i) {
    x = 0;
    y = 0;

    r1 = 0;
    r2 = 0;

    pthread_t t1;
    pthread_create(&t1, /*attr=*/NULL, &write_x_read_y, /*arg=*/NULL);
    pthread_t t2;
    pthread_create(&t2, /*attr=*/NULL, &write_y_read_x, /*arg=*/NULL);

    pthread_join(t1, /*thread_return=*/NULL);
    pthread_join(t2, /*thread_return=*/NULL);

    if (r1 == 0 && r2 == 0) {
      printf("Iteration #%lu: CPU is broken\n", i);
      abort();
    }
  }
}


In [22]:
!./snippets/faulty/mutex.out

OSError: [Errno 5] Input/output error

### Какая проблема в этом коде? 

In [25]:
!cat snippets/mutex-problem/problem.c 
!gcc snippets/mutex-problem/problem.c -o snippets/mutex-problem/problem.out

#include <stdio.h>
#include <stdint.h>
#include <stddef.h>
#include <stdlib.h>
#include <pthread.h>

pthread_mutex_t mutex1;
pthread_mutex_t mutex2;

void* first(void* arg) {
  pthread_mutex_lock(&mutex1);
  pthread_mutex_lock(&mutex2);
  pthread_mutex_unlock(&mutex2);
  pthread_mutex_unlock(&mutex1);

  return NULL;
}

void* second(void* arg) {
  pthread_mutex_lock(&mutex2);
  pthread_mutex_lock(&mutex1);
  pthread_mutex_unlock(&mutex1);
  pthread_mutex_unlock(&mutex2);

  return NULL;
}

int main() {
  pthread_mutex_init(&mutex1, NULL);
  pthread_mutex_init(&mutex2, NULL);
  
  for (int i = 0; i < 1000; ++i) {
    printf("%d\n", i);
    fflush(stdout);

    pthread_t t1;
    pthread_create(&t1, /*attr=*/NULL, &first, /*arg=*/NULL);
    pthread_t t2;
    pthread_create(&t2, /*attr=*/NULL, &second, /*arg=*/NULL);

    pthread_join(t1, /*thread_return=*/NULL);
    pthread_join(t2, /*thread_return=*/NULL);
  }
}


In [26]:
!./snippets/mutex-problem/problem.out

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

OSError: [Errno 5] Input/output error

Такая ситуация называется дедлоком. Дедлоков нужно избегать :)

### Вернемся к атомикам
У них есть еще несколько полезных методов, например, `wait` и `notify`. Их семантика такова:
* `wait` заставляет поток, вызвавший его, прекратить исполнение до момента, пока какой-нибудь другой поток не вызовет `notify`
* `notify_one` будит один случайный поток, который сделал `wait` на этом атомике
* `notify_all` будет все потоки, спящие на этом атомике

Однако их использование самих по себе довольно затруднительно, поскольку необходима синхронизация между засыпанием и пробуждением потоков, так как у нас нет гарантии, что поток, делающий `notify`, не совершит его до того, как ждущий поток вызовет `wait`. 

Также, обычно поток засыпает до момента выполнения некоего условия, которое должен удовлетворить поток, совершающий `notify`. То, что скрывается под условием, является разделяемым ресурсом, а значит проверка условия и его удовлетворение тоже должны быть синхронизированы между собой.

Для решения этих трудностей есть еще один инструмент синхронизации: `condvar (conditional variable)`. Он используется в связке с мьютексом и имеет такую семантику:
* `wait` освобождает мьютекс и засыпает до момента, пока другой поток не вызовет `notify` на этом кондваре. При получении `notify` снова захватывает мьютекс
* `notify_one` будит один случайный поток, который сделал `wait` на этом кондваре
* `notify_all` будет все потоки, спящие на этом кондваре

Пример использования:

In [23]:
!cat snippets/condvar/condvar.c 
!gcc snippets/condvar/condvar.c -o snippets/condvar/condvar.out


#include <pthread.h>
#include <stdio.h>
#include <unistd.h>
#include <assert.h>

const size_t NUMTHREADS = 20;

int done = 0;
pthread_mutex_t mutex = PTHREAD_MUTEX_INITIALIZER;
pthread_cond_t cond = PTHREAD_COND_INITIALIZER;

void* thread_entry(void* id) {
  const int myid = (int)id;
  
  const int workloops = 5;
  for (int i = 0; i < workloops; i++) {
      printf("[thread %d] working (%d/%d)\n", myid, i, workloops);
      sleep(1); // simulate doing some costly work
    }
  
  pthread_mutex_lock(&mutex);
  done++;
  printf("[thread %d] done is now %d. Signalling cond.\n", myid, done);

  // wake up the main thread (if it is sleeping) to test the value of done  
  pthread_cond_signal(&cond); 
  pthread_mutex_unlock(&mutex);

  return NULL;
}

int main( int argc, char** argv ) {
  pthread_t threads[NUMTHREADS];

  for(int t = 0; t < NUMTHREADS; t++) {
    pthread_create(&threads[t], NULL, thread_entry, (void*)(long)t);
  }

  // we're going to test "done" so we need the mutex for safe

In [24]:
!./snippets/condvar/condvar.out

[thread 0] working (0/5)
[thread 6] working (0/5)
[thread 8] working (0/5)
[thread 9] working (0/5)
[thread 4] working (0/5)
[thread 5] working (0/5)
[thread 1] working (0/5)
[thread 16] working (0/5)
[thread 2] working (0/5)
[thread 19] working (0/5)
[thread 10] working (0/5)
[thread 11] working (0/5)
[thread 12] working (0/5)
[thread 13] working (0/5)
[thread 14] working (0/5)
[thread 15] working (0/5)
[thread 7] working (0/5)
[thread 17] working (0/5)
[thread 18] working (0/5)
[thread 3] working (0/5)
[thread main] done is 0 which is < 20 so waiting on cond
[thread 8] working (1/5)
[thread 10] working (1/5)
[thread 14] working (1/5)
[thread 7] working (1/5)
[thread 12] working (1/5)
[thread 18] working (1/5)
[thread 0] working (1/5)
[thread 5] working (1/5)
[thread 9] working (1/5)
[thread 16] working (1/5)
[thread 19] working (1/5)
[thread 11] working (1/5)
[thread 13] working (1/5)
[thread 15] working (1/5)
[thread 17] working (1/5)
[thread 3] working (1/5)
[thread 6] working (1/5

### Lock-freedom

Использование mutex позволяет писать довольно простой код, но у него есть 2 проблемы:
1. Это может быть не всегда эффективно
2. Каждый поток, выполняющий работу, зависит от других потоков, работающих с тем же мьютексом. Если другие потоки затягиваются, то текущий поток не сможет сделать прогресс в работе.

У атомиков есть еще одна интересная операция - Compare-and-Swap (CAS):

```c
_Bool atomic_compare_exchange_strong(_Atomic(T) *object, T *expected, T desired);
```

Она атомарно делает следующую операцию: проверяет, лежит ли в `*object` значение, равное `*expected`, и если да, то заменяет его на `desired`. Возвращает `true`, если `*object` было заменено, и `false` в противном случае.

С использованием этой операции можно реализовать много lock-free алгоритмов. Рассмотрим lock-free стек:

In [37]:
!cat snippets/lock-free/stack.c 
!gcc snippets/lock-free/stack.c -o snippets/lock-free/stack.out

#include <stdio.h>
#include <stdlib.h>
#include <stdatomic.h>
#include <stdbool.h>
#include <pthread.h>

typedef struct stack_node {
  int payload;
  struct stack_node* prev;
} stack_node_t;

typedef struct lf_mpmc_stack {
  struct stack_node* _Atomic top;
} lf_mpmc_stack;

void stack_init(lf_mpmc_stack* stack) {
  stack->top = NULL;
}

void stack_destroy(lf_mpmc_stack* stack) {
  stack_node_t* curr = stack->top;
  while (curr != NULL) {
    stack_node_t* prev = curr->prev;
    free(curr);
    curr = prev;
  }
}

void stack_push(lf_mpmc_stack* stack, int value) {
  stack_node_t* node = malloc(sizeof(stack_node_t));
  node->payload = value;

  while (true) {
    stack_node_t* top = atomic_load(&stack->top);
    node->prev = top;
    if (atomic_compare_exchange_weak(&stack->top, &top, node)) {
      return;
    }
  }
}

stack_node_t* stack_pop(lf_mpmc_stack* stack) {
  stack_node_t* top;
  while ((top = atomic_load(&stack->top)) != NULL) {
    if (atomic_compare_exchange_weak(&stack->top

In [40]:
!./snippets/lock-free/stack.out | tail -n 100

got: 100
got: 99
got: 98
got: 97
got: 96
got: 95
got: 94
got: 93
got: 92
got: 91
got: 90
got: 89
got: 88
got: 87
got: 86
got: 85
got: 84
got: 83
got: 82
got: 81
got: 80
got: 79
got: 78
got: 77
got: 76
got: 75
got: 74
got: 73
got: 72
got: 71
got: 70
got: 69
got: 68
got: 67
got: 66
got: 65
got: 64
got: 63
got: 62
got: 61
got: 60
got: 59
got: 58
got: 57
got: 56
got: 55
got: 54
got: 53
got: 52
got: 51
got: 50
got: 49
got: 48
got: 47
got: 46
got: 45
got: 44
got: 43
got: 42
got: 41
got: 40
got: 39
got: 38
got: 37
got: 36
got: 35
got: 34
got: 33
got: 32
got: 31
got: 30
got: 29
got: 28
got: 27
got: 26
got: 25
got: 24
got: 23
got: 22
got: 21
got: 20
got: 19
got: 18
got: 17
got: 16
got: 15
got: 14
got: 13
got: 12
got: 11
got: 10
got: 9
got: 7
got: 6
got: 5
got: 4
got: 3
got: 2
got: 1
got: 0


Эта реализация подвержена [проблеме A-B-A](https://gitlab.carzil.ru/mipt-os-basic/lectures/-/blob/main/07-synchronization-basics/main.md?ref_type=heads)